In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_err = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (df_metric, df_trace, df_log, df_err):
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to produce compact list representations (limit 50)
def compact_list(values, limit=50):
    uniq = list(pd.unique(values))
    if len(uniq) > limit:
        return f"{uniq[:limit]} (+{len(uniq)-limit} more)"
    else:
        return uniq

# Summaries
summary_metric = pd.DataFrame([{
    'rows': len(df_metric),
    'min_ts': df_metric['ts'].min(),
    'max_ts': df_metric['ts'].max(),
    'unique_cmdb_ids': compact_list(df_metric['cmdb_id']),
    'unique_kpi_names': compact_list(df_metric['kpi_name'])
}])

summary_trace = pd.DataFrame([{
    'rows': len(df_trace),
    'min_ts': df_trace['ts'].min(),
    'max_ts': df_trace['ts'].max(),
    'unique_cmdb_ids': compact_list(df_trace['cmdb_id']),
    'unique_trace_names': compact_list(df_trace['trace_name'])
}])

summary_log = pd.DataFrame([{
    'rows': len(df_log),
    'min_ts': df_log['ts'].min(),
    'max_ts': df_log['ts'].max(),
    'unique_cmdb_ids': compact_list(df_log['cmdb_id']),
    'unique_log_names': compact_list(df_log['log_name'])
}])

summary_error_logs = pd.DataFrame([{
    'rows': len(df_err),
    'min_ts': df_err['ts'].min() if 'ts' in df_err else None,
    'max_ts': df_err['ts'].max() if 'ts' in df_err else None,
    'unique_cmdb_ids': compact_list(df_err['cmdb_id']),
    'unique_messages': compact_list(df_err['message'])
}])

# For metric.csv: top 20 (cmdb_id, kpi_name) pairs by row count
metric_top_pairs = (
    df_metric.groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Display compact summaries and top pairs for metric (multiple variables separated by commas)
summary_metric, metric_top_pairs, summary_trace, summary_log, summary_error_logs

```
Out[1]:
```
```python
# Build a plain-English summary using previously loaded summary variables
rows_m = int(summary_metric.at[0, 'rows'])
min_m = summary_metric.at[0, 'min_ts']
max_m = summary_metric.at[0, 'max_ts']
uniq_cmdb_m = summary_metric.at[0, 'unique_cmdb_ids']
uniq_kpis_m = summary_metric.at[0, 'unique_kpi_names']

pairs = [
    f"{r['cmdb_id']} / {r['kpi_name']} = {r['count']}"
    for _, r in metric_top_pairs.iterrows()
]
pairs_text = "; ".join(pairs)

rows_t = int(summary_trace.at[0, 'rows'])
min_t = summary_trace.at[0, 'min_ts']
max_t = summary_trace.at[0, 'max_ts']
uniq_cmdb_t = summary_trace.at[0, 'unique_cmdb_ids']
uniq_traces = summary_trace.at[0, 'unique_trace_names']

rows_l = int(summary_log.at[0, 'rows'])
min_l = summary_log.at[0, 'min_ts']
max_l = summary_log.at[0, 'max_ts']
uniq_cmdb_l = summary_log.at[0, 'unique_cmdb_ids']
uniq_logs = summary_log.at[0, 'unique_log_names']

rows_e = int(summary_error_logs.at[0, 'rows'])
min_e = summary_error_logs.at[0, 'min_ts']
max_e = summary_error_logs.at[0, 'max_ts']
uniq_cmdb_e = summary_error_logs.at[0, 'unique_cmdb_ids']

summary = (
    "Summary of telemetry files:\n\n"
    f"metric.csv:\n"
    f"- rows: {rows_m}\n"
    f"- time range (UTC): {min_m} to {max_m}\n"
    f"- unique cmdb_id (sample/limited): {uniq_cmdb_m}\n"
    f"- unique kpi_name (sample/limited): {uniq_kpis_m}\n"
    f"- top 20 (cmdb_id / kpi_name = count): {pairs_text}\n\n"
    f"trace.csv:\n"
    f"- rows: {rows_t}\n"
    f"- time range (UTC): {min_t} to {max_t}\n"
    f"- unique cmdb_id (sample/limited): {uniq_cmdb_t}\n"
    f"- unique trace_name (sample/limited): {uniq_traces}\n\n"
    f"log.csv:\n"
    f"- rows: {rows_l}\n"
    f"- time range (UTC): {min_l} to {max_l}\n"
    f"- unique cmdb_id (sample/limited): {uniq_cmdb_l}\n"
    f"- unique log_name (sample/limited): {uniq_logs}\n\n"
    f"error_logs.csv:\n"
    f"- rows: {rows_e}\n"
    f"- time range (UTC): {min_e} to {max_e}\n"
    f"- unique cmdb_id (sample/limited): {uniq_cmdb_e}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   rows                    min_ts                    max_ts                                    unique_cmdb_ids                                   unique_kpi_names
0  2000 2024-01-19 04:43:00+00:00 2024-01-19 05:07:00+00:00  [carts, carts-db, catalogue, catalogue-db, fro...  [cpu, error, latency-50, latency-90, mem, sock...,          cmdb_id    kpi_name  count
0          carts         cpu     25
1          carts       error     25
2          carts  latency-50     25
3          carts  latency-90     25
4          carts         mem     25
5          carts      socket     25
6          carts    workload     25
7       carts-db         cpu     25
8       carts-db      diskio     25
9       carts-db         mem     25
10      carts-db      socket     25
11     catalogue         cpu     25
12     catalogue      diskio     25
13     catalogue       error     25
14     catalogue  latency-50     25
15     catalogue  latency-90     25
16     catalogue         mem     25
17     catalogue      socket     25
18     catalogue    workload     25
19  catalogue-db         cpu     25,    rows min_ts max_ts unique_cmdb_ids unique_trace_names
0     0    NaT    NaT              []                 [],    rows                    min_ts                    max_ts                                    unique_cmdb_ids                  unique_log_names
0   452 2024-01-19 04:43:00+00:00 2024-01-19 05:07:00+00:00  [carts-db, catalogue, front-end, orders, payme...  [log.error_count, log.row_count],    rows min_ts max_ts unique_cmdb_ids unique_messages
0     0    NaT    NaT              []              [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure ts column exists and is UTC datetime
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Compute global quantiles (P95, P90, P50, P15, P05) per (cmdb_id, kpi_name) across entire series
grouped = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
p95 = grouped.quantile(0.95).rename('p95')
p90 = grouped.quantile(0.90).rename('p90')
p50 = grouped.quantile(0.50).rename('p50')
p15 = grouped.quantile(0.15).rename('p15')
p05 = grouped.quantile(0.05).rename('p05')

thresholds = pd.concat([p95, p90, p50, p15, p05], axis=1).reset_index()

# 2) Filter metric.csv to incident window (inclusive)
start = pd.to_datetime("2024-01-19 04:40:34", utc=True)
end = pd.to_datetime("2024-01-19 05:10:34", utc=True)
df_in = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()

# 3) Determine anomaly direction per KPI name
# Tokens indicating larger-is-worse
larger_worse_tokens = ['cpu', 'mem', 'latency', 'diskio', 'socket', 'error', 'err']  # latency covers latency-50/90
# Tokens indicating drop-is-worse (throughput/traffic)
lower_worse_tokens = ['workload', 'success', 'throughput', 'requests', 'rps']

def infer_direction(kpi):
    k = str(kpi).lower()
    if any(tok in k for tok in lower_worse_tokens):
        return 'lower_worse'
    if any(tok in k for tok in larger_worse_tokens):
        return 'larger_worse'
    # default: treat larger as worse (conservative for anomaly detection)
    return 'larger_worse'

thresholds['direction'] = thresholds['kpi_name'].apply(infer_direction)

# Merge thresholds into in-window data for row-wise anomaly checks
df_in = df_in.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left', suffixes=('', '_th'))

# Compute row-level anomaly flags
# For larger-worse: anomaly if value > p95 (primary). We'll also compute > p90 for fallback.
df_in['anomaly_p95'] = False
df_in['anomaly_p90'] = False
df_in['below_p05'] = False
df_in['below_p15'] = False

mask_larger = df_in['direction'] == 'larger_worse'
mask_lower = df_in['direction'] == 'lower_worse'

df_in.loc[mask_larger, 'anomaly_p95'] = df_in.loc[mask_larger, 'value'] > df_in.loc[mask_larger, 'p95']
df_in.loc[mask_larger, 'anomaly_p90'] = df_in.loc[mask_larger, 'value'] > df_in.loc[mask_larger, 'p90']

df_in.loc[mask_lower, 'below_p05'] = df_in.loc[mask_lower, 'value'] < df_in.loc[mask_lower, 'p05']
df_in.loc[mask_lower, 'below_p15'] = df_in.loc[mask_lower, 'value'] < df_in.loc[mask_lower, 'p15']

# Aggregate per pair within the window
agg = df_in.groupby(['cmdb_id', 'kpi_name', 'direction']).agg(
    total_points_in_window=('value','size'),
    max_value_in_window=('value','max'),
    count_p95=('anomaly_p95','sum'),
    count_p90=('anomaly_p90','sum'),
    count_below_p05=('below_p05','sum'),
    count_below_p15=('below_p15','sum'),
    earliest_p95_ts=('ts', lambda s: s[df_in.loc[s.index,'anomaly_p95']].min() if (df_in.loc[s.index,'anomaly_p95']).any() else pd.NaT),
    earliest_below_p05_ts=('ts', lambda s: s[df_in.loc[s.index,'below_p05']].min() if (df_in.loc[s.index,'below_p05']).any() else pd.NaT)
).reset_index()

# Merge back global thresholds (p95/p50) for reporting
agg = agg.merge(thresholds[['cmdb_id','kpi_name','p95','p50','p90','p05','p15','direction']], on=['cmdb_id','kpi_name','direction'], how='left')

# Determine final anomaly_count and earliest anomalous timestamp and threshold used
def determine_anomaly_row(r):
    if r['direction'] == 'larger_worse':
        if r['count_p95'] > 0:
            return pd.Series({
                'anomaly_count': int(r['count_p95']),
                'earliest_anomalous_timestamp': r['earliest_p95_ts'],
                'threshold_used': 'P95'
            })
        elif r['count_p90'] > 0:
            # fallback to P90 if no P95 breaches
            # earliest timestamp for P90:
            return pd.Series({
                'anomaly_count': int(r['count_p90']),
                'earliest_anomalous_timestamp': (df_in[(df_in['cmdb_id']==r['cmdb_id']) & (df_in['kpi_name']==r['kpi_name']) & (df_in['anomaly_p90'])]['ts'].min() if r['count_p90']>0 else pd.NaT),
                'threshold_used': 'P90'
            })
        else:
            return pd.Series({'anomaly_count': 0, 'earliest_anomalous_timestamp': pd.NaT, 'threshold_used': 'P95'})
    else:  # lower_worse
        if r['count_below_p05'] > 0:
            return pd.Series({
                'anomaly_count': int(r['count_below_p05']),
                'earliest_anomalous_timestamp': r['earliest_below_p05_ts'],
                'threshold_used': 'P05'
            })
        elif r['count_below_p15'] > 0:
            # use P15 as weaker signal if no P05 breaches
            first_ts = (df_in[(df_in['cmdb_id']==r['cmdb_id']) & (df_in['kpi_name']==r['kpi_name']) & (df_in['below_p15'])]['ts'].min()
                        if r['count_below_p15']>0 else pd.NaT)
            return pd.Series({
                'anomaly_count': int(r['count_below_p15']),
                'earliest_anomalous_timestamp': first_ts,
                'threshold_used': 'P15'
            })
        else:
            return pd.Series({'anomaly_count': 0, 'earliest_anomalous_timestamp': pd.NaT, 'threshold_used': 'P05'})

det = agg.apply(determine_anomaly_row, axis=1)
agg = pd.concat([agg, det], axis=1)

# Keep requested columns and formatting
result_df = agg[['cmdb_id','kpi_name','anomaly_count','total_points_in_window',
                 'earliest_anomalous_timestamp','max_value_in_window','p95','p50','threshold_used']].copy()

# Sort by anomaly_count desc, then total_points_in_window desc
result_df = result_df.sort_values(['anomaly_count','total_points_in_window'], ascending=[False, False]).reset_index(drop=True)

# Limit to top 20
result_df = result_df.head(20)

# Identify pairs that had zero points in-window (present in global thresholds but not in df_in)
all_pairs = thresholds.set_index(['cmdb_id','kpi_name']).index
in_window_pairs = df_in.set_index(['cmdb_id','kpi_name']).index.unique()
zero_in_window_idx = [p for p in all_pairs if p not in in_window_pairs]
# Prepare up to 10 examples
zero_pairs = pd.DataFrame(zero_in_window_idx, columns=['cmdb_id','kpi_name']).head(10)

# Final compact outputs: aggregate table and short note (zero_pairs up to 10)
result_df, zero_pairs

```
Out[2]:
```
# Build a plain-English compact summary based on the previously computed result_df and zero_pairs
top = result_df.head(10)  # top entries already sorted by anomaly_count
lines = ["Telemetry anomaly summary (metric.csv, incident window 2024-01-19 04:40:34 to 2024-01-19 05:10:34 UTC):", ""]

if top.shape[0] == 0:
    lines.append("No anomalous (cmdb_id, kpi_name) pairs detected in-window.")
else:
    lines.append(f"Top {min(10, top.shape[0])} anomalous (cmdb_id, kpi_name) pairs:")
    for i, row in top.reset_index(drop=True).iterrows():
        cmdb = row['cmdb_id']
        kpi = row['kpi_name']
        cnt = int(row['anomaly_count'])
        total = int(row['total_points_in_window'])
        ts = row['earliest_anomalous_timestamp']
        ts_text = ts.isoformat() if pd.notna(ts) else "N/A"
        maxv = float(row['max_value_in_window']) if pd.notna(row['max_value_in_window']) else None
        p95 = float(row['p95']) if pd.notna(row['p95']) else None
        p50 = float(row['p50']) if pd.notna(row['p50']) else None
        thr = row['threshold_used']
        lines.append(
            f"{i+1}. {cmdb} / {kpi}: {cnt} anomalous point(s) out of {total}; "
            f"earliest anomalous ts {ts_text}; max_in_window={maxv:.6g}; "
            f"global_P95={p95:.6g}, global_P50={p50:.6g}; threshold used={thr}"
        )

# Note about pairs with zero in-window points
if zero_pairs.shape[0] == 0:
    lines.append("")
    lines.append("All (cmdb_id, kpi_name) pairs have at least one data point inside the incident window.")
else:
    examples = [f"{r['cmdb_id']}/{r['kpi_name']}" for _, r in zero_pairs.iterrows()]
    lines.append("")
    lines.append(f"Pairs with zero data points in-window (showing up to {len(examples)}): " + ", ".join(examples))

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  anomaly_count  total_points_in_window earliest_anomalous_timestamp  max_value_in_window           p95           p50 threshold_used
0          carts         cpu              2                      25    2024-01-19 04:43:00+00:00         1.587530e+00  1.546450e+00  1.451831e+00            P95
1          carts  latency-50              2                      25    2024-01-19 04:43:00+00:00         1.093917e+00  5.882590e-01  3.180037e-01            P95
2          carts  latency-90              2                      25    2024-01-19 04:43:00+00:00         4.505070e+00  2.827275e+00  1.756658e+00            P95
3          carts         mem              2                      25    2024-01-19 05:04:00+00:00         2.195367e+08  2.194840e+08  2.194166e+08            P95
4          carts      socket              2                      25    2024-01-19 04:43:00+00:00         5.100000e+01  5.049000e+01  4.595000e+01            P95
5          carts    workload              2                      25    2024-01-19 04:57:00+00:00         8.235962e+00  8.190290e+00  7.658950e+00            P05
6       carts-db         cpu              2                      25    2024-01-19 04:43:00+00:00         4.214397e+01  3.649392e+01  3.202013e+01            P95
7       carts-db      diskio              2                      25    2024-01-19 04:43:00+00:00         3.435220e+06  3.402077e+06  3.074778e+06            P95
8       carts-db         mem              2                      25    2024-01-19 05:06:00+00:00         2.227695e+08  2.226349e+08  2.211649e+08            P95
9      catalogue         cpu              2                      25    2024-01-19 04:59:00+00:00         2.261628e-01  2.207517e-01  1.748399e-01            P95
10     catalogue  latency-50              2                      25    2024-01-19 05:02:00+00:00         5.856397e-01  5.807425e-01  3.141946e-03            P95
11     catalogue  latency-90              2                      25    2024-01-19 05:02:00+00:00         1.424021e+00  1.386776e+00  1.281326e-01            P95
12     catalogue         mem              2                      25    2024-01-19 05:01:00+00:00         6.312346e+06  6.305109e+06  6.211516e+06            P95
13     catalogue      socket              2                      25    2024-01-19 05:02:00+00:00         1.060000e+01  1.057333e+01  6.500000e+00            P95
14     catalogue    workload              2                      25    2024-01-19 04:57:00+00:00         4.144567e+00  4.088263e+00  3.818767e+00            P05
15  catalogue-db         cpu              2                      25    2024-01-19 05:00:00+00:00         2.612790e-01  2.533878e-01  2.076688e-01            P95
16  catalogue-db         mem              2                      25    2024-01-19 05:04:00+00:00         4.324886e+08  4.324570e+08  4.324361e+08            P95
17  catalogue-db      socket              2                      25    2024-01-19 05:00:00+00:00         4.716667e+00  4.513333e+00  4.000000e+00            P95
18     front-end         cpu              2                      25    2024-01-19 04:48:00+00:00         4.209897e+00  4.186761e+00  4.015186e+00            P95
19     front-end  latency-50              2                      25    2024-01-19 05:00:00+00:00         5.478995e-02  5.303844e-02  4.787898e-02            P95, Empty DataFrame
Columns: [cmdb_id, kpi_name]
Index: [])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_log if present, else load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure ts column as UTC datetimes
if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# 1) Compute global thresholds per (cmdb_id, log_name) across full series
grp = df_log.groupby(['cmdb_id', 'log_name'])['value']
th_p95 = grp.quantile(0.95).rename('p95')
th_p15 = grp.quantile(0.15).rename('p15')
th_p05 = grp.quantile(0.05).rename('p05')

thresholds_df = pd.concat([th_p95, th_p15, th_p05], axis=1).reset_index()

# 2) Filter to incident window (inclusive)
start = pd.to_datetime("2024-01-19 04:40:34", utc=True)
end = pd.to_datetime("2024-01-19 05:10:34", utc=True)
df_log_in = df_log[(df_log['ts'] >= start) & (df_log['ts'] <= end)].copy()

# Merge thresholds into in-window data
df_log_in = df_log_in.merge(thresholds_df, on=['cmdb_id','log_name'], how='left')

# Initialize flags
df_log_in['breach_p95'] = False
df_log_in['breach_p05'] = False
df_log_in['breach_p15'] = False

# Apply breach logic per log_name
mask_error = df_log_in['log_name'] == 'log.error_count'
mask_row = df_log_in['log_name'] == 'log.row_count'

df_log_in.loc[mask_error, 'breach_p95'] = df_log_in.loc[mask_error, 'value'] > df_log_in.loc[mask_error, 'p95']
df_log_in.loc[mask_row, 'breach_p05'] = df_log_in.loc[mask_row, 'value'] <= df_log_in.loc[mask_row, 'p05']
df_log_in.loc[mask_row, 'breach_p15'] = df_log_in.loc[mask_row, 'value'] <= df_log_in.loc[mask_row, 'p15']

# Aggregate per (cmdb_id, log_name)
agg = df_log_in.groupby(['cmdb_id','log_name']).agg(
    total_points_in_window=('value','size'),
    count_p95=('breach_p95','sum'),
    count_p05=('breach_p05','sum'),
    count_p15=('breach_p15','sum'),
    earliest_p95_ts=('ts', lambda s: s[df_log_in.loc[s.index,'breach_p95']].min() if (df_log_in.loc[s.index,'breach_p95']).any() else pd.NaT),
    earliest_p05_ts=('ts', lambda s: s[df_log_in.loc[s.index,'breach_p05']].min() if (df_log_in.loc[s.index,'breach_p05']).any() else pd.NaT),
    earliest_p15_ts=('ts', lambda s: s[df_log_in.loc[s.index,'breach_p15']].min() if (df_log_in.loc[s.index,'breach_p15']).any() else pd.NaT),
    max_in_window=('value', 'max'),
    min_in_window=('value', 'min'),
).reset_index()

# Merge global thresholds
agg = agg.merge(thresholds_df, on=['cmdb_id','log_name'], how='left')

# Determine reported anomaly_count, earliest timestamp, extremal value, threshold used/value
def decide_row(r):
    if r['log_name'] == 'log.error_count':
        if r['count_p95'] > 0:
            return pd.Series({
                'anomaly_count': int(r['count_p95']),
                'earliest_anomalous_timestamp': r['earliest_p95_ts'],
                'extremal_value_in_window': r['max_in_window'],
                'threshold_used': 'P95',
                'threshold_value': r['p95']
            })
        else:
            return pd.Series({
                'anomaly_count': 0,
                'earliest_anomalous_timestamp': pd.NaT,
                'extremal_value_in_window': r['max_in_window'],
                'threshold_used': 'P95',
                'threshold_value': r['p95']
            })
    elif r['log_name'] == 'log.row_count':
        # primary is P05 (<=), fallback P15
        if r['count_p05'] > 0:
            return pd.Series({
                'anomaly_count': int(r['count_p05']),
                'earliest_anomalous_timestamp': r['earliest_p05_ts'],
                'extremal_value_in_window': r['min_in_window'],
                'threshold_used': 'P05',
                'threshold_value': r['p05']
            })
        elif r['count_p15'] > 0:
            return pd.Series({
                'anomaly_count': int(r['count_p15']),
                'earliest_anomalous_timestamp': r['earliest_p15_ts'],
                'extremal_value_in_window': r['min_in_window'],
                'threshold_used': 'P15',
                'threshold_value': r['p15']
            })
        else:
            return pd.Series({
                'anomaly_count': 0,
                'earliest_anomalous_timestamp': pd.NaT,
                'extremal_value_in_window': r['min_in_window'],
                'threshold_used': 'P05',
                'threshold_value': r['p05']
            })
    else:
        # default safe output
        return pd.Series({
            'anomaly_count': 0,
            'earliest_anomalous_timestamp': pd.NaT,
            'extremal_value_in_window': np.nan,
            'threshold_used': None,
            'threshold_value': np.nan
        })

dec = agg.apply(decide_row, axis=1)
report = pd.concat([agg[['cmdb_id','log_name','total_points_in_window']], dec], axis=1)

# Select and order columns as requested
report = report[['cmdb_id','log_name','anomaly_count','total_points_in_window',
                 'earliest_anomalous_timestamp','extremal_value_in_window',
                 'threshold_used','threshold_value']]

# Sort by anomaly_count desc, then total_points desc and limit to top 20
report = report.sort_values(['anomaly_count','total_points_in_window'], ascending=[False, False]).reset_index(drop=True).head(20)

# 3) In-window rows for 'carts' and 'catalogue' for both log.error_count and log.row_count, limited to 20 rows each
wanted = df_log_in[df_log_in['cmdb_id'].isin(['carts','catalogue']) & df_log_in['log_name'].isin(['log.error_count','log.row_count'])].copy()
wanted = wanted[['ts','cmdb_id','log_name','value']].sort_values(['cmdb_id','log_name','ts'])

carts_logs = wanted[wanted['cmdb_id']=='carts'].head(20).reset_index(drop=True)
catalogue_logs = wanted[wanted['cmdb_id']=='catalogue'].head(20).reset_index(drop=True)

# Outputs: thresholds_df (stored), report (aggregated anomalies top20), carts_logs, catalogue_logs
thresholds_df, report, carts_logs, catalogue_logs

```
Out[3]:
```
# Build a plain-English summary from previously computed data (thresholds_df, report, df_log_in, carts_logs, catalogue_logs)
import pandas as pd
# Use existing variables from kernel: thresholds_df, report, df_log_in, carts_logs, catalogue_logs

# 1) Global threshold observations
err_p95 = thresholds_df[thresholds_df['log_name']=='log.error_count']['p95'].unique()
all_err_p95_zero = (len(err_p95) > 0) and (err_p95.min() == 0 and err_p95.max() == 0)

row_threshold_examples = thresholds_df[thresholds_df['log_name']=='log.row_count'][['cmdb_id','p05','p15','p95']].head(6)

# 2) Top anomaly summary (from report)
top_anoms = report[report['anomaly_count']>0].copy().head(12)
top_lines = []
for _, r in top_anoms.iterrows():
    ts = r['earliest_anomalous_timestamp']
    ts_text = ts.isoformat() if pd.notna(ts) else "N/A"
    top_lines.append(
        f"{r['cmdb_id']} / {r['log_name']}: {int(r['anomaly_count'])} anomalous pts of {int(r['total_points_in_window'])}; "
        f"earliest={ts_text}; extremal_value={r['extremal_value_in_window']}; threshold={r['threshold_used']}={r['threshold_value']}"
    )

# 3) carts & catalogue in-window raw inspection summary
def inspect_component_rows(df_in, comp):
    sub = df_in[(df_in['cmdb_id']==comp) & df_in['log_name'].isin(['log.error_count','log.row_count'])]
    summary = []
    for ln in ['log.error_count','log.row_count']:
        s = sub[sub['log_name']==ln]
        if s.empty:
            summary.append(f"{ln}: 0 rows in-window")
            continue
        cnt = len(s)
        # extremal and timestamp
        if ln == 'log.error_count':
            maxv = s['value'].max()
            ts = s[s['value']==maxv]['ts'].min()
            summary.append(f"{ln}: {cnt} rows; max={int(maxv)} at {ts.isoformat()}")
        else:
            minv = s['value'].min()
            ts = s[s['value']==minv]['ts'].min()
            summary.append(f"{ln}: {cnt} rows; min={minv} at {ts.isoformat()}")
    return summary

carts_inspect = inspect_component_rows(df_log_in, 'carts')
catalogue_inspect = inspect_component_rows(df_log_in, 'catalogue')

# Compose final summary string
lines = []
lines.append("Log analysis summary (incident window 2024-01-19 04:40:34 to 2024-01-19 05:10:34 UTC):")
lines.append("")
# Global thresholds
lines.append("Global thresholds (computed from full series):")
if all_err_p95_zero:
    lines.append("- log.error_count: global P95 = 0 for all components (no high error-count baseline).")
else:
    lines.append("- log.error_count: P95 varies across components.")
lines.append("- Example log.row_count thresholds (P05 / P15 / P95) for a few components:")
for _, row in row_threshold_examples.iterrows():
    lines.append(f"  * {row['cmdb_id']}: P05={row['p05']}, P15={row['p15']}, P95={row['p95']}")
lines.append("")

# Anomaly aggregation
lines.append("Anomalies detected in-window (top entries):")
if len(top_lines)==0:
    lines.append("- No threshold breaches detected in-window.")
else:
    for L in top_lines:
        lines.append("- " + L)
lines.append("")

# Note about error_count breaches
err_breaches = report[(report['log_name']=='log.error_count') & (report['anomaly_count']>0)]
if err_breaches.empty:
    lines.append("No log.error_count breaches were found in-window for any component (all observed error counts <= global P95).")
else:
    lines.append("Some log.error_count breaches occurred (see top list).")
lines.append("")

# carts and catalogue raw rows inspection
lines.append("In-window raw log rows (inspection) for carts and catalogue:")
lines.append(f"- carts: {carts_inspect[0]}; {carts_inspect[1]}")
lines.append(f"- catalogue: {catalogue_inspect[0]}; {catalogue_inspect[1]}")
lines.append("")

# Short actionable note
lines.append("Actionable notes:")
lines.append("- The primary anomalies are low log.row_count (traffic drops) on several services (session-db, carts-db, catalogue, front-end, orders, payment, queue-master, shipping, user).")
lines.append("- No spikes in log.error_count were observed; error metrics remain at zero baseline.")
lines.append("- Inspect the affected services' incoming traffic and downstream dependencies (session-db and carts-db show strongest row_count breaches).")

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id         log_name     p95    p15    p05
0       carts  log.error_count    0.00    0.0    0.0
1       carts    log.row_count  101.25   50.0   25.0
2    carts-db  log.error_count    0.00    0.0    0.0
3    carts-db    log.row_count  403.60  180.6  176.2
4   catalogue  log.error_count    0.00    0.0    0.0
..        ...              ...     ...    ...    ...
17   shipping    log.row_count   62.00   53.6   36.0
18       user  log.error_count    0.00    0.0    0.0
19       user    log.row_count  590.40  522.0  364.8
20    user-db  log.error_count    0.00    0.0    0.0
21    user-db    log.row_count    1.00    1.0    1.0

[22 rows x 5 columns],          cmdb_id         log_name  anomaly_count  total_points_in_window earliest_anomalous_timestamp  extremal_value_in_window threshold_used  threshold_value
0     session-db    log.row_count              5                       5    2024-01-19 04:47:00+00:00                         5            P05              5.0
1       carts-db    log.row_count              2                      25    2024-01-19 04:43:00+00:00                        89            P05            176.2
2      catalogue    log.row_count              2                      25    2024-01-19 04:43:00+00:00                        68            P05             98.6
3      front-end    log.row_count              2                      25    2024-01-19 04:43:00+00:00                       706            P05           1233.8
4         orders    log.row_count              2                      25    2024-01-19 04:43:00+00:00                        52            P05             72.0
5        payment    log.row_count              2                      25    2024-01-19 04:43:00+00:00                        44            P05             61.6
6   queue-master    log.row_count              2                      25    2024-01-19 04:43:00+00:00                       234            P05            324.0
7       shipping    log.row_count              2                      25    2024-01-19 04:43:00+00:00                        26            P05             36.0
8           user    log.row_count              2                      25    2024-01-19 04:43:00+00:00                       232            P05            364.8
9          carts    log.row_count              2                      20    2024-01-19 04:49:00+00:00                        25            P05             25.0
10       user-db    log.row_count              1                       1    2024-01-19 04:44:00+00:00                         1            P05              1.0
11      carts-db  log.error_count              0                      25                          NaT                         0            P95              0.0
12     catalogue  log.error_count              0                      25                          NaT                         0            P95              0.0
13     front-end  log.error_count              0                      25                          NaT                         0            P95              0.0
14        orders  log.error_count              0                      25                          NaT                         0            P95              0.0
15       payment  log.error_count              0                      25                          NaT                         0            P95              0.0
16  queue-master  log.error_count              0                      25                          NaT                         0            P95              0.0
17      shipping  log.error_count              0                      25                          NaT                         0            P95              0.0
18          user  log.error_count              0                      25                          NaT                         0            P95              0.0
19         carts  log.error_count              0                      20                          NaT                         0            P95              0.0,                           ts cmdb_id         log_name  value
0  2024-01-19 04:44:00+00:00   carts  log.error_count      0
1  2024-01-19 04:45:00+00:00   carts  log.error_count      0
2  2024-01-19 04:46:00+00:00   carts  log.error_count      0
3  2024-01-19 04:47:00+00:00   carts  log.error_count      0
4  2024-01-19 04:49:00+00:00   carts  log.error_count      0
5  2024-01-19 04:50:00+00:00   carts  log.error_count      0
6  2024-01-19 04:51:00+00:00   carts  log.error_count      0
7  2024-01-19 04:52:00+00:00   carts  log.error_count      0
8  2024-01-19 04:53:00+00:00   carts  log.error_count      0
9  2024-01-19 04:55:00+00:00   carts  log.error_count      0
10 2024-01-19 04:56:00+00:00   carts  log.error_count      0
11 2024-01-19 04:57:00+00:00   carts  log.error_count      0
12 2024-01-19 04:58:00+00:00   carts  log.error_count      0
13 2024-01-19 04:59:00+00:00   carts  log.error_count      0
14 2024-01-19 05:00:00+00:00   carts  log.error_count      0
15 2024-01-19 05:01:00+00:00   carts  log.error_count      0
16 2024-01-19 05:03:00+00:00   carts  log.error_count      0
17 2024-01-19 05:05:00+00:00   carts  log.error_count      0
18 2024-01-19 05:06:00+00:00   carts  log.error_count      0
19 2024-01-19 05:07:00+00:00   carts  log.error_count      0,                           ts    cmdb_id         log_name  value
0  2024-01-19 04:43:00+00:00  catalogue  log.error_count      0
1  2024-01-19 04:44:00+00:00  catalogue  log.error_count      0
2  2024-01-19 04:45:00+00:00  catalogue  log.error_count      0
3  2024-01-19 04:46:00+00:00  catalogue  log.error_count      0
4  2024-01-19 04:47:00+00:00  catalogue  log.error_count      0
5  2024-01-19 04:48:00+00:00  catalogue  log.error_count      0
6  2024-01-19 04:49:00+00:00  catalogue  log.error_count      0
7  2024-01-19 04:50:00+00:00  catalogue  log.error_count      0
8  2024-01-19 04:51:00+00:00  catalogue  log.error_count      0
9  2024-01-19 04:52:00+00:00  catalogue  log.error_count      0
10 2024-01-19 04:53:00+00:00  catalogue  log.error_count      0
11 2024-01-19 04:54:00+00:00  catalogue  log.error_count      0
12 2024-01-19 04:55:00+00:00  catalogue  log.error_count      0
13 2024-01-19 04:56:00+00:00  catalogue  log.error_count      0
14 2024-01-19 04:57:00+00:00  catalogue  log.error_count      0
15 2024-01-19 04:58:00+00:00  catalogue  log.error_count      0
16 2024-01-19 04:59:00+00:00  catalogue  log.error_count      0
17 2024-01-19 05:00:00+00:00  catalogue  log.error_count      0
18 2024-01-19 05:01:00+00:00  catalogue  log.error_count      0
19 2024-01-19 05:02:00+00:00  catalogue  log.error_count      0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['carts','catalogue','orders','payment','user']

# Incident window (UTC)
start = pd.to_datetime("2024-01-19 04:40:34", utc=True)
end = pd.to_datetime("2024-01-19 05:10:34", utc=True)

# 1) Global thresholds for metric.csv (P95, P50, and P15/P05 when relevant) computed over full series
grp = df_metric.groupby(['cmdb_id','kpi_name'])['value']
metric_p95 = grp.quantile(0.95).rename('p95')
metric_p50 = grp.quantile(0.50).rename('p50')
metric_p15 = grp.quantile(0.15).rename('p15')
metric_p05 = grp.quantile(0.05).rename('p05')

metric_thresholds = pd.concat([metric_p95, metric_p50, metric_p15, metric_p05], axis=1).reset_index()

# Restrict thresholds to candidate components only (but keep their KPIs)
metric_thresholds_cand = metric_thresholds[metric_thresholds['cmdb_id'].isin(candidates)].copy()

# Helper to infer direction: 'larger_worse' or 'lower_worse' (traffic)
larger_worse_tokens = ['cpu','mem','latency','diskio','socket','error','err']
lower_worse_tokens = ['workload','success','throughput','requests','rps']

def infer_direction(kpi):
    k = str(kpi).lower()
    if any(tok in k for tok in lower_worse_tokens):
        return 'lower_worse'
    if any(tok in k for tok in larger_worse_tokens):
        return 'larger_worse'
    # default conservative: treat larger as worse
    return 'larger_worse'

metric_thresholds_cand['direction'] = metric_thresholds_cand['kpi_name'].apply(infer_direction)

# 2) Filter metric.csv to incident window
df_metric_in = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()

# Merge thresholds into in-window metric rows (so comparisons use global thresholds)
df_metric_in = df_metric_in.merge(metric_thresholds_cand, on=['cmdb_id','kpi_name'], how='right', suffixes=('','_th'))

# After merge, some rows may be NaN in value where there were no in-window points - keep them for reporting zeros
# For convenience, ensure numeric columns present
for col in ['value','p95','p50','p15','p05']:
    if col not in df_metric_in.columns:
        df_metric_in[col] = np.nan

# Mark anomaly flags based on direction and global thresholds (computed earlier)
df_metric_in['anom_p95'] = False
df_metric_in['anom_p05'] = False

mask_larger = df_metric_in['direction'] == 'larger_worse'
mask_lower = df_metric_in['direction'] == 'lower_worse'

# For rows with actual metric values, set flags
df_metric_in.loc[mask_larger & df_metric_in['value'].notna(), 'anom_p95'] = df_metric_in.loc[mask_larger & df_metric_in['value'].notna(), 'value'] > df_metric_in.loc[mask_larger & df_metric_in['value'].notna(), 'p95']
df_metric_in.loc[mask_lower & df_metric_in['value'].notna(), 'anom_p05'] = df_metric_in.loc[mask_lower & df_metric_in['value'].notna(), 'value'] <= df_metric_in.loc[mask_lower & df_metric_in['value'].notna(), 'p05']

# 3) Aggregate per (cmdb_id,kpi_name) for candidate set
agg_metric = df_metric_in.groupby(['cmdb_id','kpi_name','direction','p95','p50','p05']).agg(
    total_points_in_window=('value', lambda s: int(s.count())),  # count non-NaN values
    anomaly_count_p95=('anom_p95','sum'),
    anomaly_count_p05=('anom_p05','sum'),
    earliest_anom_p95_ts=('ts', lambda s: s[df_metric_in.loc[s.index,'anom_p95']].min() if (df_metric_in.loc[s.index,'anom_p95']).any() else pd.NaT),
    earliest_anom_p05_ts=('ts', lambda s: s[df_metric_in.loc[s.index,'anom_p05']].min() if (df_metric_in.loc[s.index,'anom_p05']).any() else pd.NaT),
    max_value_in_window=('value','max'),
    min_value_in_window=('value','min')
).reset_index()

# Determine final anomaly_count and earliest_anomalous_timestamp based on direction
def finalize_metric_row(r):
    if r['direction'] == 'larger_worse':
        anom_cnt = int(r['anomaly_count_p95'])
        earliest = r['earliest_anom_p95_ts']
        thr_used = 'P95'
    else:
        anom_cnt = int(r['anomaly_count_p05'])
        earliest = r['earliest_anom_p05_ts']
        thr_used = 'P05'
    # severity_ratio
    sev = np.nan
    if r['direction'] == 'larger_worse':
        if pd.notna(r['p95']) and r['p95'] > 0 and pd.notna(r['max_value_in_window']):
            sev = (r['max_value_in_window'] - r['p95']) / r['p95']
    else:
        if pd.notna(r['p05']) and r['p05'] > 0 and pd.notna(r['min_value_in_window']):
            sev = (r['p05'] - r['min_value_in_window']) / r['p05']
    return pd.Series({
        'anomaly_count': anom_cnt,
        'earliest_anomalous_timestamp': earliest,
        'threshold_used_metric': thr_used,
        'severity_ratio': sev
    })

final_metric = agg_metric.join(agg_metric.apply(finalize_metric_row, axis=1))

# Keep requested metric columns
metric_report = final_metric[['cmdb_id','kpi_name','anomaly_count','total_points_in_window',
                              'earliest_anomalous_timestamp','max_value_in_window','min_value_in_window',
                              'p95','p50','p05','severity_ratio']].copy()

# 4) Log.csv analysis: compute global P05 for log.row_count and P95 for log.error_count (full series), then in-window counts for candidates
log_grp = df_log.groupby(['cmdb_id','log_name'])['value']
log_p95 = log_grp.quantile(0.95).rename('p95')
log_p05 = log_grp.quantile(0.05).rename('p05')
log_thresholds = pd.concat([log_p95, log_p05], axis=1).reset_index()

# Filter logs to incident window
df_log_in = df_log[(df_log['ts'] >= start) & (df_log['ts'] <= end)].copy()
# Merge thresholds
df_log_in = df_log_in.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')

# For candidate components, aggregate log anomalies
logs_cand = df_log_in[df_log_in['cmdb_id'].isin(candidates)].copy()

# Prepare per-component log summary
def summarize_logs_for_component(comp):
    sub = logs_cand[logs_cand['cmdb_id']==comp]
    total_points = len(sub)
    # row_count anomalies (<=P05)
    rc = sub[sub['log_name']=='log.row_count'].copy()
    if not rc.empty:
        p05 = rc['p05'].iloc[0]
        row_count_anom_mask = rc['value'] <= p05
        row_count_anom_cnt = int(row_count_anom_mask.sum())
        earliest_row_count_anom_ts = rc.loc[row_count_anom_mask,'ts'].min() if row_count_anom_cnt>0 else pd.NaT
        extremal_row_count_in_window = rc['value'].min()
    else:
        row_count_anom_cnt = 0
        earliest_row_count_anom_ts = pd.NaT
        extremal_row_count_in_window = np.nan
        p05 = np.nan
    # error_count anomalies (>P95)
    ec = sub[sub['log_name']=='log.error_count'].copy()
    if not ec.empty:
        p95 = ec['p95'].iloc[0]
        error_anom_mask = ec['value'] > p95
        error_anom_cnt = int(error_anom_mask.sum())
        earliest_error_anom_ts = ec.loc[error_anom_mask,'ts'].min() if error_anom_cnt>0 else pd.NaT
    else:
        error_anom_cnt = 0
        earliest_error_anom_ts = pd.NaT
        p95 = np.nan
    return pd.Series({
        'total_log_points_in_window': int(total_points),
        'log_row_count_anomaly_count': int(row_count_anom_cnt),
        'earliest_log_row_count_anom_ts': earliest_row_count_anom_ts,
        'log_row_count_extremal_in_window': extremal_row_count_in_window,
        'log_row_count_global_P05': p05,
        'log_error_count_anomaly_count': int(error_anom_cnt),
        'earliest_log_error_count_anom_ts': earliest_error_anom_ts,
        'log_error_count_global_P95': p95
    })

log_summaries = pd.DataFrame([summarize_logs_for_component(c) for c in candidates], index=candidates).reset_index().rename(columns={'index':'cmdb_id'})

# 5) Combine metric_report rows (for candidate components) with log summaries
combined = metric_report.merge(log_summaries, on='cmdb_id', how='left')

# Ensure ordering and fill NaNs sensibly
combined = combined.sort_values(['cmdb_id','anomaly_count'], ascending=[True, False]).reset_index(drop=True)

# Limit output to at most 50 rows
combined_out = combined.head(50)

# Additionally compute earliest anomaly timestamp per component (metric or log)
# For metric: earliest anomalous timestamp per component across its KPIs
metric_earliest = metric_report.copy()
metric_earliest['earliest_anomalous_timestamp'] = pd.to_datetime(metric_earliest['earliest_anomalous_timestamp'])
metric_min_by_comp = metric_earliest.groupby('cmdb_id')['earliest_anomalous_timestamp'].min().reset_index().rename(columns={'earliest_anomalous_timestamp':'metric_earliest_anom_ts'})

# For logs: earliest between row_count and error_count anomalies we computed
log_earliest = log_summaries[['cmdb_id','earliest_log_row_count_anom_ts','earliest_log_error_count_anom_ts']].copy()
# compute min non-NaT
def min_ts(row):
    t1 = row['earliest_log_row_count_anom_ts']
    t2 = row['earliest_log_error_count_anom_ts']
    ts = pd.to_datetime([t for t in [t1,t2] if pd.notna(t)])
    return ts.min() if len(ts)>0 else pd.NaT

log_earliest['log_earliest_anom_ts'] = log_earliest.apply(min_ts, axis=1)
log_min_by_comp = log_earliest[['cmdb_id','log_earliest_anom_ts']]

# Merge metric and log earliest and pick overall earliest per component
earliest_comb = pd.merge(metric_min_by_comp, log_min_by_comp, on='cmdb_id', how='outer')
def pick_earliest(row):
    a = row['metric_earliest_anom_ts']
    b = row['log_earliest_anom_ts']
    cand = pd.to_datetime([t for t in [a,b] if pd.notna(t)])
    return cand.min() if len(cand)>0 else pd.NaT

earliest_comb['overall_earliest_anom_ts'] = earliest_comb.apply(pick_earliest, axis=1)
earliest_summary = earliest_comb[['cmdb_id','overall_earliest_anom_ts']].copy()

# Final compact outputs
combined_out, earliest_summary

```
Out[4]:
```
Summary (plain English) — candidate services: carts, catalogue, orders, payment, user

Overall findings
- All five services show metric threshold breaches during the incident window (2024-01-19 04:40:34 → 2024-01-19 05:10:34 UTC). Breaches are few per KPI (typically 2 anomalous points per KPI out of 25 samples in-window).
- Log analysis shows low traffic (log.row_count) anomalies for these services (usually 2 points in-window per component). No log.error_count breaches were observed (log.error_count P95 = 0 and observed values stayed at 0).
- The earliest metric anomalies for these services occur at 2024-01-19 04:43:00 UTC (per-component earliest anomaly time = 2024-01-19 04:43:00 UTC). Earliest log.row_count anomaly timestamps: carts at 2024-01-19 04:49:00 UTC; catalogue/orders/payment/user at 2024-01-19 04:43:00 UTC (as reported).

Per-component concise summary (noting the most relevant KPIs / severity)
- carts
  - Metric anomalies (each with anomaly_count = 2): cpu, latency-50 (severity_ratio ≈ 0.86 — highest), latency-90 (≈ 0.59), mem, socket, workload.
  - Log: log.row_count anomaly_count = 2 (earliest row_count anomaly 2024-01-19 04:49:00 UTC; extremal row_count in-window = 25; global P05 = 25.0). log.error_count anomaly_count = 0.
  - Earliest anomaly overall: 2024-01-19 04:43:00 UTC.

- catalogue
  - Metric anomalies (each anomaly_count = 2): cpu, latency-50 (notable), latency-90, mem, socket, workload.
  - Log: log.row_count anomaly_count = 2 (earliest 2024-01-19 04:43:00 UTC; extremal row_count in-window = 68; global P05 ≈ 98.6). log.error_count anomaly_count = 0.
  - Earliest anomaly overall: 2024-01-19 04:43:00 UTC.

- orders
  - Metric anomalies (typical set: cpu, latency variants, mem, socket, workload) — anomaly_count usually 2 per KPI.
  - Log: log.row_count anomaly_count = 2 (earliest 2024-01-19 04:43:00 UTC). log.error_count anomaly_count = 0.
  - Earliest anomaly overall: 2024-01-19 04:43:00 UTC.

- payment
  - Metric anomalies similar to other services (cpu, latency, mem, socket, workload), with small anomaly counts (2).
  - Log: log.row_count anomaly_count = 2 (earliest 2024-01-19 04:43:00 UTC). log.error_count anomaly_count = 0.
  - Earliest anomaly overall: 2024-01-19 04:43:00 UTC.

- user
  - Metric anomalies: cpu, latency-50/90, mem, workload (anomaly_count typically 2); some KPIs show 0 anomalies (e.g., "error" KPI).
  - Log: log.row_count anomaly_count = 2 (earliest 2024-01-19 04:43:00 UTC); log.error_count anomaly_count = 0.
  - Earliest anomaly overall: 2024-01-19 04:43:00 UTC.

Interpretation / actionable points
- The dominant signal is a short, system-wide traffic drop (log.row_count breaches across services) coinciding with metric disturbances (notably latency spikes). Error counts did not rise, so failures are not driven by increased error rates.
- The most severe metric symptom observed is latency (e.g., carts latency-50 shows the largest severity ratio ~0.86), suggesting service slowdowns coincident with traffic changes.
- Earliest detected anomalies (metrics and many logs) occur at ~2024-01-19 04:43:00 UTC — this is the best timestamp to start deeper investigation (trace the request path, ingress/load-balancer, upstream dependencies, and any rate-limiting or throttling around that time).
- Recommended next steps: inspect ingress/load-balancer and network/queueing around 04:43 UTC; check dependencies that could cause broad traffic drops (e.g., session-db or auth services), and review traces for high-latency spans around that timestamp.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id    kpi_name  anomaly_count  total_points_in_window earliest_anomalous_timestamp  max_value_in_window  min_value_in_window           p95           p50           p05  severity_ratio  total_log_points_in_window  log_row_count_anomaly_count earliest_log_row_count_anom_ts  log_row_count_extremal_in_window  log_row_count_global_P05  log_error_count_anomaly_count earliest_log_error_count_anom_ts  \
0    carts         cpu              2                      25    2024-01-19 04:43:00+00:00         1.587530e+00         1.354001e+00  1.546450e+00  1.451831e+00  1.364504e+00        0.026564                          40                            2      2024-01-19 04:49:00+00:00                                25                      25.0                              0                              NaT   
1    carts  latency-50              2                      25    2024-01-19 04:43:00+00:00         1.093917e+00         1.855637e-01  5.882590e-01  3.180037e-01  2.280372e-01        0.859584                          40                            2      2024-01-19 04:49:00+00:00                                25                      25.0                              0                              NaT   
2    carts  latency-90              2                      25    2024-01-19 04:43:00+00:00         4.505070e+00         9.942645e-01  2.827275e+00  1.756658e+00  1.241550e+00        0.593432                          40                            2      2024-01-19 04:49:00+00:00                                25                      25.0                              0                              NaT   
3    carts         mem              2                      25    2024-01-19 05:04:00+00:00         2.195367e+08         2.192992e+08  2.194840e+08  2.194166e+08  2.193076e+08        0.000240                          40                            2      2024-01-19 04:49:00+00:00                                25                      25.0                              0                              NaT   
4    carts      socket              2                      25    2024-01-19 04:43:00+00:00         5.100000e+01         4.340000e+01  5.049000e+01  4.595000e+01  4.341667e+01        0.010101                          40                            2      2024-01-19 04:49:00+00:00                                25                      25.0                              0                              NaT   
..     ...         ...            ...                     ...                          ...                  ...                  ...           ...           ...           ...             ...                         ...                          ...                            ...                               ...                       ...                            ...                              ...   
31    user  latency-90              2                      25    2024-01-19 04:43:00+00:00         9.065620e-03         8.755004e-03  9.053410e-03  8.955435e-03  8.818384e-03        0.001349                          50                            2      2024-01-19 04:43:00+00:00                               232                     364.8                              0                              NaT   
32    user         mem              2                      25    2024-01-19 05:05:00+00:00         8.795819e+06         8.743385e+06  8.794085e+06  8.782916e+06  8.759706e+06        0.000197                          50                            2      2024-01-19 04:43:00+00:00                               232                     364.8                              0                              NaT   
33    user    workload              2                      25    2024-01-19 04:57:00+00:00         1.970285e+01         1.586103e+01  1.820968e+01  1.730318e+01  1.598683e+01        0.007869                          50                            2      2024-01-19 04:43:00+00:00                               232                     364.8                              0                              NaT   
34    user       error              0                      25                          NaT         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             NaN                          50                            2      2024-01-19 04:43:00+00:00                               232                     364.8                              0                              NaT   
35    user      socket              0                      25                          NaT         2.000000e+01         1.800000e+01  2.000000e+01  2.000000e+01  1.888000e+01        0.000000                          50                            2      2024-01-19 04:43:00+00:00                               232                     364.8                              0                              NaT   

    log_error_count_global_P95  
0                          0.0  
1                          0.0  
2                          0.0  
3                          0.0  
4                          0.0  
..                         ...  
31                         0.0  
32                         0.0  
33                         0.0  
34                         0.0  
35                         0.0  

[36 rows x 19 columns],      cmdb_id  overall_earliest_anom_ts
0      carts 2024-01-19 04:43:00+00:00
1  catalogue 2024-01-19 04:43:00+00:00
2     orders 2024-01-19 04:43:00+00:00
3    payment 2024-01-19 04:43:00+00:00
4       user 2024-01-19 04:43:00+00:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if available, else load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Candidate services and incident window
candidates = ['carts','catalogue','orders','payment','user']
start = pd.to_datetime("2024-01-19 04:40:34", utc=True)
end = pd.to_datetime("2024-01-19 05:10:34", utc=True)

# 1) Global metric thresholds (computed on full series BEFORE filtering)
metric_grp = df_metric.groupby(['cmdb_id','kpi_name'])['value']
m_p95 = metric_grp.quantile(0.95).rename('p95')
m_p50 = metric_grp.quantile(0.50).rename('p50')
m_p15 = metric_grp.quantile(0.15).rename('p15')
m_p05 = metric_grp.quantile(0.05).rename('p05')
metric_thresholds = pd.concat([m_p95, m_p50, m_p15, m_p05], axis=1).reset_index()

# Infer direction for KPIs
larger_worse_tokens = ['cpu','mem','latency','diskio','socket','error','err']
lower_worse_tokens = ['workload','success','throughput','requests','rps']
def infer_direction(kpi):
    k = str(kpi).lower()
    if any(tok in k for tok in lower_worse_tokens):
        return 'lower_worse'
    if any(tok in k for tok in larger_worse_tokens):
        return 'larger_worse'
    return 'larger_worse'
metric_thresholds['direction'] = metric_thresholds['kpi_name'].apply(infer_direction)

# Keep only candidate services' KPIs
metric_thresholds_cand = metric_thresholds[metric_thresholds['cmdb_id'].isin(candidates)].copy()

# 2) Filter metric to incident window
df_metric_in = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()

# 3) For each (cmdb_id,kpi_name) in candidate set compute anomalies using global thresholds
# Prepare a DataFrame of all candidate pairs (from thresholds) to ensure pairs with zero in-window rows are included
pairs = metric_thresholds_cand[['cmdb_id','kpi_name','p95','p50','p05','p15','direction']].copy()

# Function to compute per-pair in-window aggregates
def compute_pair_stats(row):
    cmdb = row['cmdb_id']; kpi = row['kpi_name']
    p95 = row['p95']; p50 = row['p50']; p05 = row['p05']
    direction = row['direction']
    sub = df_metric_in[(df_metric_in['cmdb_id']==cmdb) & (df_metric_in['kpi_name']==kpi)]
    total_pts = int(sub['value'].count())
    if total_pts == 0:
        return pd.Series({
            'total_points_in_window': 0,
            'anomaly_count': 0,
            'earliest_anomalous_timestamp': pd.NaT,
            'max_value_in_window': np.nan,
            'min_value_in_window': np.nan,
            'p95': p95, 'p50': p50, 'p05': p05,
            'severity_ratio': np.nan
        })
    max_v = sub['value'].max()
    min_v = sub['value'].min()
    if direction == 'larger_worse':
        anom_mask = sub['value'] > p95
        anom_cnt = int(anom_mask.sum())
        earliest = sub.loc[anom_mask, 'ts'].min() if anom_cnt>0 else pd.NaT
        # severity ratio (use p95)
        sev = np.nan
        if pd.notna(p95) and p95>0:
            sev = (max_v - p95)/p95
    else:
        anom_mask = sub['value'] <= p05
        anom_cnt = int(anom_mask.sum())
        earliest = sub.loc[anom_mask, 'ts'].min() if anom_cnt>0 else pd.NaT
        sev = np.nan
        if pd.notna(p05) and p05>0:
            sev = (p05 - min_v)/p05
    return pd.Series({
        'total_points_in_window': total_pts,
        'anomaly_count': anom_cnt,
        'earliest_anomalous_timestamp': earliest,
        'max_value_in_window': max_v,
        'min_value_in_window': min_v,
        'p95': p95, 'p50': p50, 'p05': p05,
        'severity_ratio': sev
    })

pair_stats = pairs.apply(compute_pair_stats, axis=1)
pair_report = pd.concat([pairs.reset_index(drop=True), pair_stats.reset_index(drop=True)], axis=1)

# Aggregate per service
agg_service = pair_report.groupby('cmdb_id').agg(
    overall_earliest_metric_anom_ts=('earliest_anomalous_timestamp', lambda s: pd.to_datetime(s.dropna()).min() if s.notna().any() else pd.NaT),
    max_severity_ratio=('severity_ratio', lambda s: float(s.dropna().max()) if s.notna().any() else np.nan),
    total_metric_anomaly_count=('anomaly_count','sum'),
    metric_kpis_with_anom=('anomaly_count', lambda s: int((s>0).sum()))
).reset_index()

# 4) Log thresholds (global) and in-window aggregates for candidates
log_grp = df_log.groupby(['cmdb_id','log_name'])['value']
log_p95 = log_grp.quantile(0.95).rename('p95')
log_p05 = log_grp.quantile(0.05).rename('p05')
log_thresholds = pd.concat([log_p95, log_p05], axis=1).reset_index()

# Filter logs to window
df_log_in = df_log[(df_log['ts'] >= start) & (df_log['ts'] <= end)].copy()

# For each candidate service compute required log fields
def compute_log_stats_for_service(comp):
    # row_count
    rc_thr = log_thresholds[(log_thresholds['cmdb_id']==comp) & (log_thresholds['log_name']=='log.row_count')]
    if not rc_thr.empty:
        rc_p05 = rc_thr['p05'].iloc[0]
    else:
        rc_p05 = np.nan
    rc_sub = df_log_in[(df_log_in['cmdb_id']==comp) & (df_log_in['log_name']=='log.row_count')]
    if not rc_sub.empty:
        rc_min = rc_sub['value'].min()
        rc_anom_mask = rc_sub['value'] <= rc_p05 if pd.notna(rc_p05) else pd.Series([False]*len(rc_sub))
        rc_anom_cnt = int(rc_anom_mask.sum())
        rc_earliest = rc_sub.loc[rc_anom_mask,'ts'].min() if rc_anom_cnt>0 else pd.NaT
    else:
        rc_min = np.nan
        rc_anom_cnt = 0
        rc_earliest = pd.NaT
    # error_count
    ec_thr = log_thresholds[(log_thresholds['cmdb_id']==comp) & (log_thresholds['log_name']=='log.error_count')]
    if not ec_thr.empty:
        ec_p95 = ec_thr['p95'].iloc[0]
    else:
        ec_p95 = np.nan
    ec_sub = df_log_in[(df_log_in['cmdb_id']==comp) & (df_log_in['log_name']=='log.error_count')]
    if not ec_sub.empty:
        ec_anom_mask = ec_sub['value'] > ec_p95 if pd.notna(ec_p95) else pd.Series([False]*len(ec_sub))
        ec_anom_cnt = int(ec_anom_mask.sum())
        ec_earliest = ec_sub.loc[ec_anom_mask,'ts'].min() if ec_anom_cnt>0 else pd.NaT
    else:
        ec_anom_cnt = 0
        ec_earliest = pd.NaT
    return pd.Series({
        'log_row_count_anomaly_count': int(rc_anom_cnt),
        'earliest_log_row_count_anom_ts': rc_earliest,
        'log_row_count_min_in_window': rc_min,
        'log_error_count_anomaly_count': int(ec_anom_cnt),
        'log_error_count_global_P95': ec_p95
    })

log_summaries = pd.DataFrame([compute_log_stats_for_service(c) for c in candidates], index=candidates).reset_index().rename(columns={'index':'cmdb_id'})

# 5) Combine metric agg and log summaries for candidate services
combined = pd.merge(agg_service, log_summaries, on='cmdb_id', how='right')
# Ensure all candidate services present; fill missing metric aggregates with defaults
combined = combined[['cmdb_id','overall_earliest_metric_anom_ts','max_severity_ratio','total_metric_anomaly_count','metric_kpis_with_anom',
                     'log_row_count_anomaly_count','earliest_log_row_count_anom_ts','log_row_count_min_in_window',
                     'log_error_count_anomaly_count','log_error_count_global_P95']]

# Sort by max_severity_ratio desc (NaNs last), then overall_earliest_metric_anom_ts asc
combined_sorted = combined.sort_values(by=['max_severity_ratio','overall_earliest_metric_anom_ts'],
                                       ascending=[False, True], na_position='last').reset_index(drop=True)

# Limit to the 5 candidate services
final_table = combined_sorted[combined_sorted['cmdb_id'].isin(candidates)].reset_index(drop=True)

# One-line summary: service with highest max_severity_ratio and its earliest anomaly timestamp
if final_table['max_severity_ratio'].notna().any():
    top_row = final_table.loc[final_table['max_severity_ratio'].idxmax()]
    top_service = top_row['cmdb_id']
    top_ts = top_row['overall_earliest_metric_anom_ts']
    summary_line = f"Highest severity: {top_service} (max_severity_ratio={top_row['max_severity_ratio']:.6g}); earliest metric anomaly at {pd.to_datetime(top_ts)}"
else:
    # fall back if all NaN
    # pick earliest by timestamp
    temp = final_table.copy()
    temp['ts_sort'] = pd.to_datetime(temp['overall_earliest_metric_anom_ts'])
    temp = temp.sort_values('ts_sort')
    if temp['ts_sort'].notna().any():
        top_service = temp.iloc[0]['cmdb_id']
        top_ts = temp.iloc[0]['overall_earliest_metric_anom_ts']
        summary_line = f"Highest severity: {top_service} (no numeric severity); earliest metric anomaly at {pd.to_datetime(top_ts)}"
    else:
        summary_line = "No metric anomalies detected for the candidate services in-window."

# Final outputs: final_table (compact) and summary_line
final_table, summary_line

```
Out[5]:
```
# Build a plain-English summary string from the computed final_table and summary_line
# Reuse variables from kernel: final_table, summary_line
import pandas as pd

rows = []
rows.append("Per-service compact summary (incident window 2024-01-19 04:40:34 → 2024-01-19 05:10:34 UTC):")
rows.append("")

for _, r in final_table.iterrows():
    cmdb = r['cmdb_id']
    ts = r['overall_earliest_metric_anom_ts']
    ts_text = pd.to_datetime(ts).isoformat() if pd.notna(ts) else "N/A"
    max_sev = r['max_severity_ratio']
    max_sev_text = f"{max_sev:.6g}" if pd.notna(max_sev) else "NaN"
    total_anoms = int(r['total_metric_anomaly_count'])
    kpis_with = int(r['metric_kpis_with_anom'])
    rc_anoms = int(r['log_row_count_anomaly_count'])
    rc_earliest = r['earliest_log_row_count_anom_ts']
    rc_earliest_text = pd.to_datetime(rc_earliest).isoformat() if pd.notna(rc_earliest) else "N/A"
    rc_min = r['log_row_count_min_in_window']
    ec_anoms = int(r['log_error_count_anomaly_count'])
    ec_p95 = r['log_error_count_global_P95']
    rows.append(f"- {cmdb}: earliest_metric_anom={ts_text}; max_severity_ratio={max_sev_text}; "
                f"total_metric_anomaly_count={total_anoms}; metric_kpis_with_anom={kpis_with}; "
                f"log_row_count_anomaly_count={rc_anoms} (earliest={rc_earliest_text}, min={rc_min}); "
                f"log_error_count_anomaly_count={ec_anoms}, log_error_count_global_P95={ec_p95}")

rows.append("")
rows.append(summary_line)

summary = "\n".join(rows)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id overall_earliest_metric_anom_ts  max_severity_ratio  total_metric_anomaly_count  metric_kpis_with_anom  log_row_count_anomaly_count earliest_log_row_count_anom_ts  log_row_count_min_in_window  log_error_count_anomaly_count  log_error_count_global_P95
0      carts       2024-01-19 04:43:00+00:00            0.859584                          12                      6                            2      2024-01-19 04:49:00+00:00                           25                              0                         0.0
1     orders       2024-01-19 04:43:00+00:00            0.390848                          12                      6                            2      2024-01-19 04:43:00+00:00                           52                              0                         0.0
2  catalogue       2024-01-19 04:57:00+00:00            0.049050                          12                      6                            2      2024-01-19 04:43:00+00:00                           68                              0                         0.0
3    payment       2024-01-19 04:50:00+00:00            0.020932                          10                      5                            2      2024-01-19 04:43:00+00:00                           44                              0                         0.0
4       user       2024-01-19 04:43:00+00:00            0.011133                          10                      5                            2      2024-01-19 04:43:00+00:00                          232                              0                         0.0, 'Highest severity: carts (max_severity_ratio=0.859584); earliest metric anomaly at 2024-01-19 04:43:00+00:00')```
```